In [ ]:
import matplotlib.pyplot as plt
import os.path
import seaborn as sns
import torch

from pathlib import Path

from pyro_cases.run import vae_dict

In [ ]:
def drop_not_converge_cases(data):
    new_favi_test_dict_list = []
    new_elbo_test_dict_list = []
    num_favi_not_converge = 0
    num_elbo_not_converge = 0
    for td in data:
        if "favi_cant_converge" in td["favi_test_dict_list"]:
            num_favi_not_converge += 1
        else:
            new_favi_test_dict_list.append(td["favi_test_dict_list"])
        
        if "elbo_cant_converge" in td["elbo_test_dict_list"]:
            num_elbo_not_converge += 1
        else:
            new_elbo_test_dict_list.append(td["elbo_test_dict_list"])
    return new_favi_test_dict_list, num_favi_not_converge, new_elbo_test_dict_list, num_elbo_not_converge

In [ ]:
def load_data(data_path):
    data = torch.load(data_path, map_location=torch.device("cpu"))
    model_num = len(data)
    task_name = data[0]["task"]
    obs_num = 10
    theta_dim = vae_dict[task_name].theta_dim

    favi_est_mu = torch.full((model_num, obs_num, theta_dim), torch.nan)
    favi_est_sigma2 = torch.full((model_num, obs_num, theta_dim), torch.nan)
    elbo_est_mu = torch.full((model_num, obs_num, theta_dim), torch.nan)
    elbo_est_sigma2 = torch.full((model_num, obs_num, theta_dim), torch.nan)

    favi_test_dict_list, num_favi_not_converge, elbo_test_dict_list, num_elbo_not_converge = drop_not_converge_cases(data)
    assert len(favi_test_dict_list) > 0
    for i, tdl in enumerate(favi_test_dict_list):
        for j, td in enumerate(tdl):
            favi_est_mu[i, j] = td["est_mu"]
            favi_est_sigma2[i, j] = td["est_sigma2"]
    for i, tdl in enumerate(elbo_test_dict_list):
        for j, td in enumerate(tdl):
            elbo_est_mu[i, j] = td["est_mu"]
            elbo_est_sigma2[i, j] = td["est_sigma2"]

    true_theta = torch.zeros(obs_num, theta_dim)
    for obs_index in range(obs_num):
        true_theta[obs_index] = favi_test_dict_list[0][obs_index]["true_theta"]
    return {
        "task_name": task_name,
        "obs_num": obs_num,
        "theta_dim": theta_dim,
        "favi_est_mu": favi_est_mu,
        "favi_est_sigma2": favi_est_sigma2,
        "num_favi_not_converge": num_favi_not_converge,
        "elbo_est_mu": elbo_est_mu,
        "elbo_est_sigma2": elbo_est_sigma2,
        "num_elbo_not_converge": num_elbo_not_converge,
        "true_theta": true_theta,
    }

In [ ]:
def nanstd(tensor, dim=None, keepdim=False):
    tensor_mean = tensor.nanmean(dim=dim, keepdim=True)
    output = (tensor - tensor_mean).square().nanmean(dim=dim, keepdim=keepdim)
    return output.sqrt()

In [ ]:
def test_data_std(data_path):
    if not os.path.isfile(data_path):
        print(f"skip {data_path}")
        return None
    
    data_dict = load_data(data_path)
    favi_est_mu_std = nanstd(data_dict["favi_est_mu"], dim=0).mean()
    favi_est_sigma2_std = nanstd(data_dict["favi_est_sigma2"], dim=0).mean()
    elbo_est_mu_std = nanstd(data_dict["elbo_est_mu"], dim=0).mean()
    elbo_est_sigma2_std = nanstd(data_dict["elbo_est_sigma2"], dim=0).mean()

    return {
        "favi_est_mu_std": favi_est_mu_std, 
        "favi_est_sigma2_std": favi_est_sigma2_std, 
        "num_favi_not_converge": data_dict["num_favi_not_converge"],
        "elbo_est_mu_std": elbo_est_mu_std, 
        "elbo_est_sigma2_std": elbo_est_sigma2_std,
        "num_elbo_not_converge": data_dict["num_elbo_not_converge"],
    }

In [ ]:
data_path = Path("/data/scratch/pduan/gcvi_03-23_output")
lr_schedulers = ["plain", 
                "custom_decrease", "exponential", "milestones",
                "cosine_annealing", "cyclic", "one_cycle", 
                "cosine_annealing_warm_restart"]
network_widths = [512, 1024, 2048]
favi_mu_std = torch.zeros(len(network_widths), len(lr_schedulers))
favi_sigma2_std = torch.zeros(len(network_widths), len(lr_schedulers))
elbo_mu_std = torch.zeros(len(network_widths), len(lr_schedulers))
elbo_sigma2_std = torch.zeros(len(network_widths), len(lr_schedulers))
for i, nw in enumerate(network_widths):
    for j, lr_s in enumerate(lr_schedulers):
        tag = f"pyro_t_gaussian_linear_lr_{lr_s}_nw_{nw}_mn_100.pt"
        data_dict = test_data_std(data_path / tag)
        favi_mu_std[i, j] = data_dict["favi_est_mu_std"]
        elbo_mu_std[i, j] = data_dict["elbo_est_mu_std"]
        favi_sigma2_std[i, j] = data_dict["favi_est_sigma2_std"]
        elbo_sigma2_std[i, j] = data_dict["elbo_est_sigma2_std"]

In [ ]:
task_name = "gaussian_linear"

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(favi_mu_std,
            cmap="RdBu", center=0,
            linewidths=0.5, square=False,
            annot=True,
            fmt=".1e", cbar=True,
            yticklabels=network_widths,
            xticklabels=lr_schedulers)
plt.xlabel("LR Scheduler")
plt.ylabel("Network Width")
plt.title(f"[{task_name.upper()}] " + r"FAVI std($\mu)$")
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(elbo_mu_std - favi_mu_std,
            cmap="RdBu", center=0,
            linewidths=0.5, square=False,
            annot=True,
            fmt=".1e", cbar=True,
            yticklabels=network_widths,
            xticklabels=lr_schedulers)
plt.xlabel("LR Scheduler")
plt.ylabel("Network Width")
plt.title(f"[{task_name.upper()}] " + r"ELBO std($\mu$) - FAVI std($\mu$)")
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(favi_sigma2_std,
            cmap="RdBu", center=0,
            linewidths=0.5, square=False,
            annot=True,
            fmt=".1e", cbar=True,
            yticklabels=network_widths,
            xticklabels=lr_schedulers)
plt.xlabel("LR Scheduler")
plt.ylabel("Network Width")
plt.title(f"[{task_name.upper()}] " + r"FAVI std($\sigma^2$)")
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(elbo_sigma2_std - favi_sigma2_std,
            cmap="RdBu", center=0,
            linewidths=0.5, square=False,
            annot=True,
            fmt=".1e", cbar=True,
            yticklabels=network_widths,
            xticklabels=lr_schedulers)
plt.xlabel("LR Scheduler")
plt.ylabel("Network Width")
plt.title(f"[{task_name.upper()}] " + r"ELBO std($\sigma^2$) - FAVI std($\sigma^2$)")
plt.show()

### KDE Plots

In [ ]:
def plot_data(data_path, plot_mu, kde_plot):
    if not os.path.isfile(data_path):
        print(f"skip {data_path}")
        return None
    
    data_dict = load_data(data_path)
    obs_num = data_dict["obs_num"]
    theta_dim = data_dict["theta_dim"]
    true_theta = data_dict["true_theta"]
    task_name: str = data_dict["task_name"]

    favi_est = data_dict["favi_est_mu"] if plot_mu else data_dict["favi_est_sigma2"]
    elbo_est = data_dict["elbo_est_mu"] if plot_mu else data_dict["elbo_est_sigma2"]

    fig, axes = plt.subplots(theta_dim, obs_num, 
                             figsize=(obs_num * 2.5, theta_dim * 2.5))
    
    axes: list[plt.Axes] = axes.flatten()
    for i, ax in enumerate(axes):
        ri, ci = i % obs_num, i // obs_num
        if kde_plot:
            sns.kdeplot(favi_est[:, ri, ci], fill=True, 
                        label="FAVI", color="blue", alpha=0.3,
                        # log_scale=True,
                        ax=ax)
            sns.kdeplot(elbo_est[:, ri, ci], fill=True,
                        label="ELBO", color="red", alpha=0.3,
                        # log_scale=True,
                        ax=ax)
            ax.axvline(x=true_theta[ri, ci], color="green", linestyle="dashed")
        else:
            ax.boxplot([favi_est[:, ri, ci], 
                        elbo_est[:, ri, ci]],
                        labels=["FAVI", "ELBO"])
            ax.axhline(y=true_theta[ri, ci], color="green", linestyle="dashed")
        ax.set_title(f"obs {ri}; theta {ci}")
    
    fig.tight_layout()
    fig.subplots_adjust(right=0.8, top=theta_dim / (0.5 + theta_dim))
    fig.suptitle(task_name.replace("_", " ").upper(), 
                 fontsize=12)
    fig.legend(["FAVI", "ELBO"], 
               loc="center right", ncol=1, fontsize=12,
               bbox_to_anchor=(0.85, 0.5))
    fig.show()

In [84]:
for item in os.listdir("/data/scratch/pduan/gcvi_03-25_output/"):
    full_path = os.path.join("/data/scratch/pduan/gcvi_03-25_output/", item)
    plot_data(full_path,
              plot_mu=True, kde_plot=True)